# Lectura de logs

Construir el proceso para actualizar diariamente la tabla llamada `tabla_reporte_bot` 
a partir de la lectura (diaria) del archivo `.log` más reciente. La tabla final debe
contener las siguientes columnas:

- id
- updated_at
- timestamp
- solicitante
- target
- acción
- sistema (sobre el que se ejecutó la acción)
- nombre completo del usuario solicitante (Nombres + Apellidos)
- nombre completo del usuario target (Nombres + Apellidos)
- oficina del usuario solicitante (Campo OFFICE del ADManager)
- oficina del usuario target (Campo OFFICE del ADManager)
- resultado

Las columna `updated_at` contiene el timestamp de cuándo se cargó cada registro y la columna "id" (UUID) con un id único que representa a cada registro.

## Requisitos

- La tabla final debe ser un `csv`que se actualiza con cada ejecución diaria.
- El proceso debe ser **idempotente**, es decir, las ejecuciones/reejecuciones diarias siempre
    deben producir exactamente el mismo resultado y no deben alterar ni duplicar datos
        en la tabla final.
- El proceso se debe poder reejecutar para cualquier fecha anterior, esto lo hace capaz de
    recuperarse en caso de que exista un problema con la data input y nos pidan corregir el
    resultado después de la ejecución diaria normal. Tomando en cuenta la idempotencia, en
    caso de correcciones por fallos, las reejecuciones solamente deberán añadir nuevos
    registros no existentes en las ejecuciones originales.
- Existen dos acciones/sistemas en los logs, hay que diseñar el proceso para que tome en
    cuenta única y exclusivamente los reseteos de usuarios de ADManager (el endpoint que es
    más o menos como: users_admin/resetuser).

MUY importante la mantenibilidad a largo plazo.

El resultado final debe ser el programa modularizado a través de archivos `.py` en la carpeta `src/`. El archivo `main.py` debe ejecutar únicamente el proceso general, mientras que `src/` debe contener los módulos a utilizar (importables con `__init__.py`).

La columna `resultado_final` debe ser un mensaje legible al humano, por ejemplo:

- Estatus `200` devuelve "Proceso exitoso."
- 404 (Not Found) se debe procesar la respuesta de ADManager para saber cuál usuario en específico fue el que no se encontró: "El usuario solicitante no se encontró en ADManager" o "El usuario objetivo no se encontró en ADManager", o "Ningún usuario se encontró en ADManager".

Códigos HTTP:

`200 OK` se retorna cuando:
Todas las validaciones son exitosas y el reseteo en ADManager sí se ejecuta
correctamente.

`202 Accepted` se retorna cuando:
El campo `OFFICE` del usuario target nos indica que pertenece a Corporativo,
por lo tanto no puede ser reseteado mediante el bot porque puede autoresetearse.
CONSEJO IMPORTANTE: para las comparaciones de cadenas de texto normalicen
sus valores removiendo acentos y dejando todo en minúsculas para evitar que,
por ejemplo, en este caso no cachen que el usuario target es de corporativo si el valor
del campo `OFFICE` es literalmente "Corporativo" (con 'C' mayúscula) o
"CORPORATIVO" (todo en mayúsculas). ADManager es un sistema muy inconsistente
y no es raro que este tipo de casos sucedan.

`403 Forbidden` se retorna cuando: Los usuarios no son de la misma oficina.
El usuario solicitante no es gerente ni administrador de sistemas, para esta validación
basta con que el campo `DESCRIPTION` de ADManager comience con las subcadenas
"gerente" o "admin".

El campo `OU_NAME` de ADManager del usuario target tiene el valor "OAT/Cedis/BY" por
lo que no puede ser reseteado mediante el bot (esta es una regla muy específica del
negocio).

`404 Not Found` se retorna cuando:
El usuario target no se encontró en ADManager
El usuario solicitante no se encontró en ADManager
Ningún usuario se encontró en ADManager

`429 Too Many Requests` se retorna cuando:
No se puede ejecutar el reseteo porque se agotaron los tokens de ADManager.

`500 Internal Server Error` se retorna cuando:
Ocurrió un error inesperado. Estos casos son CRÍTICOS porque indican que algo se
rompió totalmente, casi siempre debido a bugs en nuestra propia lógica.

`503 Service Unavailable` se retorna cuando:
Ocurrió un error en ADManager, por lo que el reseteo no se pudo ejecutar. En este
caso, debe concatenarse en el mensaje final el mensaje de error exacto que
retornó ADManager al momento de ejecutar el reseteo. El log específico de
ADManager que hay que revisar en estos casos es el que comienza con el patrón |
ADM-Raw response |... .

`504 Gateway Timeout` se retorna cuando:
Ocurrió un timeout en la comunicación con ADManager, es decir, que se tardó más de
35 segundos en respondernos por lo que se da por hecho que el reseteo no se pudo
ejecutar.

# Diseño del proceso

El proceso se implementa aquí de forma exploratoria y después se traslada tal cual a
módulos `.py` en `src/`. La arquitectura es de **tres capas**:

```
data/raw/<fecha>.log
        ↓  parseo determinista (sin reloj, sin estado)
data/processed/resetuser_<fecha>.csv → staging
        ↓  upsert idempotente por `id`
reports/tabla_reporte_bot.csv
```

El staging  no contiene `updated_at` ni ningún valor derivado del reloj, así que regenerarlo produce siempre el
mismo archivo. Eso permite:

- reprocesar un día y comparar contra el staging anterior para ver exactamente qué cambió
- reconstruir el reporte completo desde cero sin volver a tocar los `.log`
- depurar el parseo sin arrastrar la lógica de acumulación.

**Cómo se garantiza la idempotencia.** Cada operación del bot tiene un `operation_Id`
único. La columna `id` se calcula como `uuid5(NAMESPACE, operation_id)`: es un UUID
válido, único por registro y **determinista**, a diferencia de `uuid4`. El upsert
descarta cualquier fila cuyo `id` ya exista en el reporte, de modo que una reejecución
—del mismo día o de uno anterior— sólo puede añadir registros nuevos y nunca modifica
el `updated_at` de los ya cargados.

**Mapa hacia `src/`.** Cada sección corresponde a un módulo:

| Sección del notebook | Módulo destino |
|---|---|
| Configuración | `config.py` |
| Normalización de texto | `normalize.py` |
| Descubrimiento de archivos | `discovery.py` |
| Lectura multilínea | `reader.py` |
| Expresiones regulares | `parsers.py` |
| Enriquecimiento de usuarios | `enrich.py` |
| Mensajes de resultado | `outcomes.py` |
| Construcción de filas y staging | `transform.py` |
| Upsert y escritura atómica | `storage.py` |
| Orquestación | `pipeline.py` + `main.py` |

## 1. Configuración

Todas las constantes de negocio (`src/config.py`)

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import re
import unicodedata
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import parse_qs, urlparse
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd

In [2]:
def project_root(start: Path | None = None) -> Path:
    """Localiza la raíz del proyecto subiendo directorios hasta encontrar `data/raw`.

    Permite que el notebook funcione igual si se ejecuta desde `notebooks/` o desde la raíz.

    Input:
        start (Path | None): directorio desde donde empezar a subir. `None` usa el
            directorio de trabajo actual.

    Output:
        Path: ruta absoluta de la raíz del proyecto.
        Lanza `RuntimeError` si ningún ancestro contiene `data/raw`.
    """
    inicio = (start or Path.cwd()).resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / "data" / "raw").is_dir():
            return candidato
    raise RuntimeError("No se encontró la raíz del proyecto (falta data/raw)")


PROJECT_ROOT = project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_PATH = REPORTS_DIR / "tabla_reporte_bot.csv"

# --- Reglas de negocio ---------------------------------------------------
# Sólo se reportan los reseteos de ADManager; /v2/sap/register_user se ignora.
ENDPOINT_OBJETIVO = "users_admin/resetuser"
SISTEMA = "ADManager"
ACCION = "reset_password"

# uuid5 estable
UUID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "tabla_reporte_bot")

OU_BLOQUEADA = "oat/cedis/by"          # ya normalizada
MARCADOR_CORPORATIVO = "corporativo"   # ya normalizado
PREFIJOS_PRIVILEGIADOS = ("gerente", "admin")
VALORES_NULOS_AD = {"", "-", "<not set>", "none"}

# Revisar consistencia en ADManager
COMPARAR_OFICINA_POR_TOKENS = False

# Salida
COLUMNAS_REPORTE = [
    "id",
    "updated_at",
    "timestamp",
    "operation_id",
    "solicitante",
    "target",
    "accion",
    "sistema",
    "nombre_solicitante",
    "nombre_target",
    "oficina_solicitante",
    "oficina_target",
    "resultado_final",
]

# El staging no lleva updated_at (para ser determinista) y sí lleva status_http (para depurar).
COLUMNAS_STAGING = [c for c in COLUMNAS_REPORTE if c != "updated_at"] + ["status_http"]

print("Reporte final en:", REPORT_PATH.relative_to(PROJECT_ROOT))

Reporte final en: reports\tabla_reporte_bot.csv


## 2. Normalización de texto

ADManager devuelve los mismos valores con acentos, mayúsculas y espacios distintos. Toda
comparación de negocio (oficina, descripción, OU) pasa primero por `norm()`.

In [3]:
def quitar_acentos(texto: str) -> str:
    """Elimina los acentos de un texto mediante la descomposición NFKD.

    NFKD separa cada carácter acentuado en letra base + marca diacrítica y aplana las
    variantes de estilo; después se descartan las marcas combinantes.

    Input:
        texto (str): cadena original, con o sin acentos.

    Output:
        str: la misma cadena sin marcas diacríticas (conserva mayúsculas y espacios).
    """
    descompuesto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))


def norm(valor: object) -> str:
    """Normaliza un valor para poder compararlo: minúsculas, sin acentos y sin espacios
    sobrantes. Es la base de toda comparación de negocio contra datos de ADManager.

    Input:
        valor (object): cualquier valor; se convierte a `str`. `None` se trata como vacío.

    Output:
        str: texto en minúsculas, sin acentos y con espacios internos colapsados a uno.
    """
    if valor is None:
        return ""
    return " ".join(quitar_acentos(str(valor)).casefold().split())


def es_valor_nulo(valor: object) -> bool:
    """Indica si un valor de ADManager representa "sin dato".

    ADManager usa indistintamente '', '-', '<not set>' y 'none' como ausencia de dato.

    Input:
        valor (object): valor crudo tal como llegó de ADManager.

    Output:
        bool: `True` si el valor normalizado está en `VALORES_NULOS_AD`.
    """
    return norm(valor) in VALORES_NULOS_AD


def misma_oficina(a: str, b: str) -> bool:
    """Determina si dos oficinas son la misma según el criterio configurado.

    Con `COMPARAR_OFICINA_POR_TOKENS = False` compara las cadenas normalizadas tal cual
    (replica el comportamiento del bot); con `True` compara los conjuntos de palabras, de
    modo que "Cedis 5687 Salinas" y "Cedis Salinas 5687" se consideran iguales.

    Input:
        a (str): oficina del usuario solicitante (campo OFFICE).
        b (str): oficina del usuario objetivo (campo OFFICE).

    Output:
        bool: `True` si ambas oficinas se consideran la misma.
    """
    na, nb = norm(a), norm(b)
    if COMPARAR_OFICINA_POR_TOKENS:
        return sorted(na.split()) == sorted(nb.split())
    return na == nb


def es_corporativo(oficina: str) -> bool:
    """Indica si una oficina pertenece a Corporativo (causa del estatus 202).

    Busca la subcadena normalizada 'corporativo', por lo que atrapa "Corporativo",
    "CORPORATIVO" y variantes con acentos o espacios extra.

    Input:
        oficina (str): valor crudo del campo OFFICE de ADManager.

    Output:
        bool: `True` si la oficina contiene el marcador de Corporativo.
    """
    return MARCADOR_CORPORATIVO in norm(oficina)


def tiene_privilegios(descripcion: str) -> bool:
    """Indica si el solicitante es gerente o administrador de sistemas.

    Basta con que el campo `DESCRIPTION` de ADManager empiece con "gerente" o "admin"
    (ya normalizado), según la regla de negocio del estatus 403.

    Input:
        descripcion (str): valor crudo del campo DESCRIPTION de ADManager.

    Output:
        bool: `True` si la descripción normalizada empieza con alguno de los prefijos de
        `PREFIJOS_PRIVILEGIADOS`.
    """
    return norm(descripcion).startswith(PREFIJOS_PRIVILEGIADOS)


# Comprobaciones rápidas de las trampas conocidas
assert norm("CORPORATIVO") == norm("Corporativo") == "corporativo"
assert es_corporativo("Oficina Corporativo")
assert es_corporativo("CORPORATIVO")
assert tiene_privilegios("Administrador De Sistemas")
assert tiene_privilegios("Gerente Tienda")
assert not tiene_privilegios("Surtidor Ropa Y Variedades")
assert es_valor_nulo("<not set>") and es_valor_nulo("-")

# Caso real de los logs: la misma oficina escrita en distinto orden.
print("Normalización OK")
print(
    "'Cedis 5687 Salinas' vs 'Cedis Salinas 5687' ->",
    "misma oficina" if misma_oficina("Cedis 5687 Salinas", "Cedis Salinas 5687") else "oficinas distintas",
    f"(COMPARAR_OFICINA_POR_TOKENS={COMPARAR_OFICINA_POR_TOKENS})",
)

Normalización OK
'Cedis 5687 Salinas' vs 'Cedis Salinas 5687' -> oficinas distintas (COMPARAR_OFICINA_POR_TOKENS=False)


## 3. Descubrimiento de archivos

Por defecto el proceso toma el `.log` más reciente (ejecución diaria). Con una fecha
explícita reprocesa cualquier día anterior (recuperación ante fallos).

In [4]:
PATRON_LOG = "*.log"


def fechas_disponibles() -> list[str]:
    """Lista las fechas con log disponible, deducidas del nombre de cada archivo.

    `Path.stem` devuelve el nombre sin extensión, que en este proyecto es directamente la
    fecha (`2026-09-01.log` -> `2026-09-01`). El orden alfabético coincide con el
    cronológico gracias al formato ISO.

    Input:
        Ninguno. Lee `RAW_DIR` con el patrón `PATRON_LOG`.

    Output:
        list[str]: fechas `YYYY-MM-DD` ordenadas de más antigua a más reciente.
    """
    return sorted(p.stem for p in RAW_DIR.glob(PATRON_LOG))


def ruta_log(fecha: str) -> Path:
    """Devuelve la ruta del `.log` crudo de una fecha concreta, validando que exista.

    Input:
        fecha (str): fecha en formato `YYYY-MM-DD`.

    Output:
        Path: ruta a `data/raw/<fecha>.log`.
        Lanza `FileNotFoundError` (con la lista de fechas disponibles) si no existe.
    """
    ruta = RAW_DIR / f"{fecha}.log"
    if not ruta.is_file():
        disponibles = ", ".join(fechas_disponibles()) or "ninguna"
        raise FileNotFoundError(f"No existe el log de {fecha}. Fechas disponibles: {disponibles}")
    return ruta


def log_mas_reciente() -> Path:
    """Devuelve la ruta del `.log` más reciente: la entrada de la ejecución diaria normal.

    Input:
        Ninguno. Se apoya en `fechas_disponibles()`.

    Output:
        Path: ruta al log de la última fecha disponible.
        Lanza `FileNotFoundError` si no hay ningún `.log` en `RAW_DIR`.
    """
    fechas = fechas_disponibles()
    if not fechas:
        raise FileNotFoundError(f"No hay archivos {PATRON_LOG} en {RAW_DIR}")
    return ruta_log(fechas[-1])


print("Fechas disponibles:", fechas_disponibles())
print("Fecha más reciente:", log_mas_reciente().name)

Fechas disponibles: ['2026-08-29', '2026-08-30', '2026-08-31', '2026-09-01']
Fecha más reciente: 2026-09-01.log


## 4. Lectura multilínea

Un registro del log no equivale a una línea del
archivo: los bloques `Raw Response: {...}` continúan en líneas *sin* prefijo de timestamp
ni `operation_Id`. Un lector línea a línea perdería justo la información de ADManager que
necesitamos (nombres, oficinas, `UsersList` vacía).

La regla: una línea que hace match con `HEADER_RE` abre un registro nuevo; cualquier otra
línea se acumula en el registro abierto.

In [ ]:
HEADER_RE = re.compile(
    r"^(?P<timestamp>\d{4}-\d{2}-\d{2}T[\d:.]+Z) \| "
    r"(?P<level>\w+) "
    r"\[operation_Id=(?P<operation_id>[0-9a-fA-F]+)\] \| "
    r"(?P<message>.*)$"
)


@dataclass
class LogRecord:
    """Un registro lógico del log: la línea con encabezado más sus continuaciones.

    Campos:
        timestamp (str): marca de tiempo ISO de la línea de encabezado.
        level (str): nivel de log (INFO, ERROR, ...).
        operation_id (str): identificador de la operación del bot.
        message (str): mensaje completo, ya con las líneas de continuación pegadas.
    """

    timestamp: str
    level: str
    operation_id: str
    message: str


@dataclass
class Operation:
    """Todos los registros que comparten un operation_Id, en orden de aparición."""

    operation_id: str
    records: list[LogRecord] = field(default_factory=list)

    @property
    def timestamp(self) -> str:
        """Marca de tiempo de la operación: la del primer registro que la abrió.

        Input:
            Ninguno (propiedad; usa `self.records`).

        Output:
            str: timestamp ISO del primer registro. Lanza `IndexError` si no hay registros.
        """
        return self.records[0].timestamp

    def messages(self) -> list[str]:
        """Devuelve los mensajes de la operación en orden de aparición.

        Es la vista que consumen los parsers, que sólo necesitan el texto.

        Input:
            Ninguno (usa `self.records`).

        Output:
            list[str]: un mensaje por registro lógico.
        """
        return [r.message for r in self.records]

    @property
    def texto_crudo(self) -> str:
        """Reconstruye el texto original de la operación tal como aparece en el `.log`.

        Se usa sólo para inspección manual en la zona de experimentación.

        Input:
            Ninguno (propiedad; usa `self.records`).

        Output:
            str: los registros concatenados con su encabezado (timestamp, nivel, operation_Id).
        """
        return "".join(
            f"{r.timestamp} | {r.level} [operation_Id={r.operation_id}] | {r.message}"
            for r in self.records
        )


def iter_log_records(path: Path):
    """Convierte las líneas físicas de un `.log` en registros lógicos multilínea.

    Punto crítico del parseo: los bloques `Raw Response: {...}` continúan en líneas sin
    encabezado. Una línea que hace match con `HEADER_RE` abre un registro nuevo; cualquier
    otra se acumula en el registro abierto.

    Input:
        path (Path): ruta al archivo `.log` crudo.

    Output:
        Iterator[LogRecord]: generador que emite un `LogRecord` por registro lógico, en
        orden de aparición.
    """
    actual: LogRecord | None = None
    buffer: list[str] = []
    with path.open(encoding="utf-8") as fh:
        for linea in fh:
            m = HEADER_RE.match(linea)
            if m:
                if actual is not None:
                    actual.message = "".join(buffer)
                    yield actual
                actual = LogRecord(m["timestamp"], m["level"], m["operation_id"], "")
                buffer = [m["message"] + "\n"]
            elif actual is not None:
                buffer.append(linea)
    if actual is not None:
        actual.message = "".join(buffer)
        yield actual


def agrupar_operaciones(path: Path) -> dict[str, Operation]:
    """Agrupa los registros de un `.log` por `operation_Id`.

    Cada operación del bot genera varias líneas (entrada, consultas a ADManager, respuesta);
    agruparlas es lo que permite resolver una fila del reporte con todo su contexto.
    Las operaciones pueden no ser consecutivas si se procesan varias al mismo tiempo.

    Input:
        path (Path): ruta al archivo `.log` crudo.

    Output:
        dict[str, Operation]: `operation_id` -> `Operation` con sus registros en orden.
    """
    operaciones: dict[str, Operation] = {}
    for rec in iter_log_records(path):
        operaciones.setdefault(rec.operation_id, Operation(rec.operation_id)).records.append(rec)
    return operaciones

In [6]:
# Comprobación: el lector multilínea recupera los bloques Raw Response
_path_demo = log_mas_reciente()
_ops_demo = agrupar_operaciones(_path_demo)

_lineas_fisicas = sum(1 for _ in _path_demo.open(encoding="utf-8"))
_registros = sum(len(o.records) for o in _ops_demo.values())

print(f"Archivo: {_path_demo.name}")
print(f"Líneas físicas: {_lineas_fisicas}")
print(f"Registros lógicos: {_registros}   <- menor: las continuaciones se absorbieron")
print(f"Operaciones únicas: {len(_ops_demo)}")

Archivo: 2026-09-01.log
Líneas físicas: 590
Registros lógicos: 350   <- menor: las continuaciones se absorbieron
Operaciones únicas: 40


## 5. Expresiones regulares

Tres patrones cubren todo lo que necesitamos. Este módulo **no conoce reglas de negocio**:
sólo extrae texto y lo convierte en estructuras de Python.

1. **`ENTRY_RE`** — la línea de entrada del bot. Trae `solicitante`, `target` y el
   **status final** de la operación, y es siempre la primera del bloque. Se distingue de
   las llamadas a ADManager porque el status va *fuera* de las comillas (`"HTTP/1.1" 404`
   contra `"HTTP/1.1 200 "`) y porque no lleva método HTTP.
2. **`SEARCH_RE`** — las consultas `SearchUser` de ADManager, con su `filter` (qué usuario
   se buscó) y su `Raw Response` en JSON. Una `UsersList` vacía es exactamente lo que
   permite resolver los sub-casos del 404.
3. **`ADM_RAW_RE`** — el bloque `| ADM-Raw response |`, del que salen el mensaje de error
   del 503 y la razón del timeout del 504.

   **log 2026-08-30**: `ADM-Raw response | status: 504`.

In [7]:
ENTRY_RE = re.compile(r"^HTTP Request: (?P<url>https?://\S+) \"HTTP/1\.1\" (?P<status>\d{3})\s*$")

SEARCH_RE = re.compile(
    r"'filter': '\((?P<campo>sAMAccountName|employeeID):equal:(?P<valor>[^)]*)\)'\},"
    r"\s*Raw Response:\s*(?P<body>.*?),\s*Raw status_code:",
    re.S,
)

ADM_RAW_RE = re.compile(r"^ADM-Raw response \| status: (?P<status>\d+) \| (?P<resto>.*)$", re.S)


def parse_entrada(message: str) -> dict | None:
    """Extrae la línea de entrada del bot: endpoint, estatus final y usuarios.

    Es la única línea que trae el status HTTP final de la operación y los dos usuarios
    involucrados, que vienen como parámetros de la query string.

    Input:
        message (str): mensaje de un `LogRecord`.

    Output:
        dict | None: `None` si la línea no es una entrada del bot; si lo es,
        `{'endpoint': str, 'status': int, 'solicitante': str, 'target': str}`.
    """
    m = ENTRY_RE.match(message.strip())
    if not m:
        return None
    url = urlparse(m["url"])
    params = parse_qs(url.query)
    return {
        "endpoint": url.path.lstrip("/"),
        "status": int(m["status"]),
        "solicitante": params.get("sAMAccountName_requester", [""])[0],
        "target": params.get("sAMAccountName_target", [""])[0],
    }


def parse_consultas_usuario(message: str) -> list[dict]:
    """Extrae las consultas `SearchUser` hechas a ADManager dentro de un mensaje.

    Una `UsersList` vacía es justamente lo que permite distinguir los sub-casos del 404
    (no se encontró el solicitante, el target o ninguno).

    Input:
        message (str): mensaje de un `LogRecord` (puede contener varias consultas).

    Output:
        list[dict]: una entrada por consulta, con
        `{'campo': str, 'valor': str, 'usuario': dict | None}`; `usuario` es el primer
        elemento de `UsersList` o `None` si la búsqueda no arrojó resultados. Las consultas
        cuyo cuerpo no es JSON válido se omiten.
    """
    resultados = []
    for m in SEARCH_RE.finditer(message):
        try:
            body = json.loads(m["body"].strip())
        except json.JSONDecodeError:
            continue
        usuarios = body.get("UsersList") or []
        resultados.append(
            {"campo": m["campo"], "valor": m["valor"], "usuario": usuarios[0] if usuarios else None}
        )
    return resultados


def parse_adm_raw(message: str) -> dict | None:
    """Extrae el bloque `| ADM-Raw response |` con la respuesta cruda de ADManager.

    De aquí salen el mensaje de error exacto del 503 y la razón del timeout del 504. El
    cuerpo viene como literal de Python (no JSON), por eso se evalúa con `ast.literal_eval`.

    Input:
        message (str): mensaje de un `LogRecord`.

    Output:
        dict | None: `None` si el mensaje no es un bloque ADM-Raw; si lo es,
        `{'status': int, 'resto': str}` más las claves opcionales `'body'` (objeto de
        Python, o `None` si no se pudo evaluar) y `'reason'` (str).
    """
    m = ADM_RAW_RE.match(message.strip())
    if not m:
        return None
    resto = m["resto"]
    info: dict = {"status": int(m["status"]), "resto": resto}
    body_m = re.match(r"body:\s*(?P<body>.*)$", resto, re.S)
    if body_m:
        try:
            info["body"] = ast.literal_eval(body_m["body"].strip())
        except (ValueError, SyntaxError):
            info["body"] = None
    reason_m = re.search(r"reason:\s*(?P<reason>[^|]+)", resto)
    if reason_m:
        info["reason"] = reason_m["reason"].strip()
    return info


def mensaje_admanager(adm: dict | None) -> str:
    """Obtiene el `statusMessage` exacto que devolvió ADManager.

    La especificación exige concatenarlo al mensaje final del estatus 503. El cuerpo puede
    venir como lista de dicts o como dict suelto, así que se contemplan ambas formas.

    Input:
        adm (dict | None): resultado de `parse_adm_raw()`, o `None`.

    Output:
        str: el `statusMessage` sin espacios sobrantes, o cadena vacía si no hay tal campo.
    """
    body = (adm or {}).get("body")
    if isinstance(body, list) and body and isinstance(body[0], dict):
        return str(body[0].get("statusMessage", "")).strip()
    if isinstance(body, dict):
        return str(body.get("statusMessage", "")).strip()
    return ""

In [17]:
# Comprobación de los tres patrones sobre una operación real
#import random
#_demo = random.choice(list(_ops_demo.values()))
_demo = next(iter(_ops_demo.values()))
print("operation_id:", _demo.operation_id)
for _rec in _demo.records:
    _e = parse_entrada(_rec.message)
    if _e:
        print("  entrada:", _e)
    for _c in parse_consultas_usuario(_rec.message):
        _u = _c["usuario"]
        print(f"  consulta: {_c['valor']:>16} -> ", "no encontrado" if _u is None
              else f"{_u['FIRST_NAME']} {_u['LAST_NAME']} | OFFICE={_u['OFFICE']}")
    _a = parse_adm_raw(_rec.message)
    if _a:
        print("  adm-raw: status", _a["status"], "|", mensaje_admanager(_a) or _a.get("reason", ""))

operation_id: b5ab6865ca8ac9629ff2eb303637d408
  entrada: {'endpoint': 'v3/users_admin/resetuser', 'status': 200, 'solicitante': 'admsistemas520', 'target': '520000228'}
  consulta:        520000228 ->  Nombre_86f0740bf6 Apellido_86f0740bf6 | OFFICE=0520
  consulta:   admsistemas520 ->  Nombre_d216ff1cda Apellido_d216ff1cda | OFFICE=0520
  adm-raw: status 200 | Password reset successful.


## 6. Enriquecimiento de usuarios

Las consultas a ADManager aparecen en orden variable (a veces primero el target, a veces
el solicitante) y a veces la búsqueda es por `employeeID` en lugar de `sAMAccountName`.
Por eso se construye un **índice** por valor normalizado, indexando tanto el valor buscado
como el `SAM_ACCOUNT_NAME` devuelto, y después se resuelven solicitante y target contra él.

Aquí también se aplica el **filtro de endpoint**: si la operación no es un
`users_admin/resetuser`, se descarta y nunca llega al reporte.

In [18]:
@dataclass
class Usuario:
    """Datos de un usuario de ADManager ya normalizados para el reporte.

    Campos:
        sam (str): `sAMAccountName` tal como lo mandó el bot.
        encontrado (bool): `False` si ADManager no devolvió al usuario (caso 404).
        nombre_completo (str): FIRST_NAME + LAST_NAME.
        oficina (str): campo OFFICE.
        descripcion (str): campo DESCRIPTION (base de la validación de privilegios).
        ou_name (str): campo OU_NAME (base de la regla de la OU bloqueada).
    """

    sam: str
    encontrado: bool = False
    nombre_completo: str = ""
    oficina: str = ""
    descripcion: str = ""
    ou_name: str = ""


@dataclass
class OperacionReset:
    """Una operación `users_admin/resetuser` ya resuelta y lista para volverse una fila.

    Campos:
        operation_id (str): identificador único de la operación en el log.
        timestamp (str): marca de tiempo de la línea de entrada del bot.
        status (int): código HTTP final devuelto por el bot.
        solicitante (Usuario): usuario que pidió el reseteo.
        target (Usuario): usuario al que se le iba a resetear la contraseña.
        adm_raw (dict | None): respuesta cruda de ADManager, necesaria para el 503/504.
    """

    operation_id: str
    timestamp: str
    status: int
    solicitante: Usuario
    target: Usuario
    adm_raw: dict | None = None


def _campo(datos: dict, clave: str) -> str:
    """Lee un campo de ADManager tratando sus marcadores de "sin dato" como cadena vacía.

    Input:
        datos (dict): diccionario del usuario devuelto por ADManager.
        clave (str): nombre del campo (`OFFICE`, `DESCRIPTION`, `OU_NAME`, ...).

    Output:
        str: el valor sin espacios sobrantes, o `''` si falta o es un valor nulo de AD.
    """
    valor = datos.get(clave, "")
    return "" if es_valor_nulo(valor) else str(valor).strip()


def construir_usuario(sam: str, datos: dict | None) -> Usuario:
    """Construye un `Usuario` a partir de la respuesta de ADManager.

    Input:
        sam (str): `sAMAccountName` del usuario, tal como lo mandó el bot.
        datos (dict | None): diccionario devuelto por ADManager, o `None` si no se encontró.

    Output:
        Usuario: con `encontrado=False` y los demás campos vacíos si `datos` es `None`; en
        caso contrario, con nombre, oficina, descripción y OU ya normalizados.
    """
    if not datos:
        return Usuario(sam=sam, encontrado=False)
    nombre = " ".join(p for p in (_campo(datos, "FIRST_NAME"), _campo(datos, "LAST_NAME")) if p)
    return Usuario(
        sam=sam,
        encontrado=True,
        nombre_completo=nombre,
        oficina=_campo(datos, "OFFICE"),
        descripcion=_campo(datos, "DESCRIPTION"),
        ou_name=_campo(datos, "OU_NAME"),
    )


def indexar_usuarios(operacion: Operation) -> dict[str, dict | None]:
    """Construye un índice de los usuarios consultados dentro de una operación.

    Las consultas aparecen en orden variable y a veces se buscan por `employeeID` en lugar
    de `sAMAccountName`, así que se indexa tanto el valor buscado como el
    `SAM_ACCOUNT_NAME` devuelto. Un hallazgo nunca se pisa con un `None`.

    Input:
        operacion (Operation): operación cruda agrupada por `operation_Id`.

    Output:
        dict[str, dict | None]: clave normalizada -> datos del usuario, o `None` si esa
        búsqueda no encontró a nadie.
    """
    indice: dict[str, dict | None] = {}
    for message in operacion.messages():
        for consulta in parse_consultas_usuario(message):
            clave = norm(consulta["valor"])
            usuario = consulta["usuario"]
            # Un hallazgo posterior nunca se pisa con un None anterior (ni al revés).
            if clave not in indice or (indice[clave] is None and usuario is not None):
                indice[clave] = usuario
            if usuario:
                indice[norm(usuario.get("SAM_ACCOUNT_NAME", ""))] = usuario
    return indice


def enriquecer(operacion: Operation) -> OperacionReset | None:
    """Resuelve una operación cruda del log en una `OperacionReset` completa.

    Recorre los registros para localizar la línea de entrada y el último bloque ADM-Raw, y
    aplica el filtro de endpoint: sólo los `users_admin/resetuser` llegan al reporte.

    Input:
        operacion (Operation): operación cruda agrupada por `operation_Id`.

    Output:
        OperacionReset | None: `None` si la operación no tiene línea de entrada o no es un
        reseteo de ADManager; en caso contrario, la operación con ambos usuarios resueltos.
    """
    entrada = None
    timestamp = operacion.timestamp
    adm_raw = None
    for rec in operacion.records:
        if entrada is None:
            candidata = parse_entrada(rec.message)
            if candidata is not None:
                entrada, timestamp = candidata, rec.timestamp
        adm = parse_adm_raw(rec.message)
        if adm is not None:
            adm_raw = adm
    if entrada is None or not entrada["endpoint"].endswith(ENDPOINT_OBJETIVO):
        return None
    indice = indexar_usuarios(operacion)
    return OperacionReset(
        operation_id=operacion.operation_id,
        timestamp=timestamp,
        status=entrada["status"],
        solicitante=construir_usuario(entrada["solicitante"], indice.get(norm(entrada["solicitante"]))),
        target=construir_usuario(entrada["target"], indice.get(norm(entrada["target"]))),
        adm_raw=adm_raw,
    )

## 7. Mensajes de resultado

Un *resolver* por código HTTP, registrados en un diccionario. Añadir un código nuevo es
añadir una función y una entrada: no se toca nada más del proceso.

Los códigos `202` y `429` **no aparecen en los logs actuales** (ver la validación más
abajo), pero se implementan igual porque son parte de la especificación y pueden aparecer
cualquier día.

### Sobre el orden de las validaciones del `403`

La especificación lista tres causas y el log no dice cuál se disparó, así que hay que
deducirla. El orden es: oficina → privilegios → OU bloqueada. Con los datos actuales las
causas resultan **disjuntas** (27 por oficina, 13 por privilegios, 0 por OU), así que el
orden no altera ningún resultado; se conserva el de la especificación por trazabilidad.

### Hallazgo: `OFFICE` inconsistente

En los logs conviven `"Cedis 5687 Salinas"` y `"Cedis Salinas 5687"` — la misma oficina
con las palabras en otro orden. La operación `56a6ad8c62501c3a1b170d67785f458b` es
justamente un solicitante de una contra un target de la otra, **y el bot devolvió 403**.
Es decir: el bot compara las cadenas sin reordenar tokens. Por eso
`COMPARAR_OFICINA_POR_TOKENS` queda en `False`: así las 41 operaciones con 403 quedan
explicadas. Poniéndolo en `True` la comparación es semánticamente "más correcta", pero esa
operación deja de tener causa identificable y cae al mensaje genérico. La tabla debe
reflejar la decisión que tomó el bot, no la que debería haber tomado.

In [19]:
def _resultado_200(op: OperacionReset) -> str:
    """Mensaje del estatus 200: todas las validaciones pasaron y el reseteo se ejecutó.

    Input:
        op (OperacionReset): operación resuelta (no se consulta ningún campo).

    Output:
        str: mensaje legible para la columna `resultado_final`.
    """
    return "Proceso exitoso."


def _resultado_202(op: OperacionReset) -> str:
    """Mensaje del estatus 202: el target es de Corporativo y puede autoresetearse.

    Input:
        op (OperacionReset): operación resuelta (no se consulta ningún campo: el estatus
            del bot ya implica la causa).

    Output:
        str: mensaje legible para la columna `resultado_final`.
    """
    return (
        "El usuario objetivo pertenece a Corporativo, "
        "por lo que puede autoresetear su contraseña sin el bot."
    )


def _resultado_403(op: OperacionReset) -> str:
    """Mensaje del estatus 403, deduciendo cuál de las tres validaciones falló.

    El log no dice qué regla se disparó, así que se evalúan en el orden de la
    especificación: oficinas distintas -> falta de privilegios -> OU bloqueada. Si ninguna
    explica el rechazo se devuelve un mensaje genérico, que en las validaciones sirve como
    alerta de que hay una causa sin modelar.

    Input:
        op (OperacionReset): operación resuelta; usa `oficina`, `descripcion` y `ou_name`.

    Output:
        str: mensaje legible con la causa identificada, o el genérico si no hay ninguna.
    """
    if not misma_oficina(op.solicitante.oficina, op.target.oficina):
        return "El usuario solicitante y el usuario objetivo no pertenecen a la misma oficina."
    if not tiene_privilegios(op.solicitante.descripcion):
        return "El usuario solicitante no es gerente ni administrador de sistemas."
    if norm(op.target.ou_name) == OU_BLOQUEADA:
        return "El usuario objetivo pertenece a la OU 'OAT/Cedis/BY' y no puede ser reseteado mediante el bot."
    return "El reseteo fue rechazado por las validaciones de acceso del bot."


def _resultado_404(op: OperacionReset) -> str:
    """Mensaje del estatus 404, distinguiendo qué usuario no se encontró en ADManager.

    La distinción sale de la bandera `encontrado`, que a su vez viene de que la `UsersList`
    de la consulta correspondiente llegara vacía.

    Input:
        op (OperacionReset): operación resuelta; usa `solicitante.encontrado` y
            `target.encontrado`.

    Output:
        str: uno de los tres mensajes de la especificación (ninguno / solicitante / target),
        o un genérico si ambos usuarios sí se encontraron.
    """
    falta_solicitante = not op.solicitante.encontrado
    falta_target = not op.target.encontrado
    if falta_solicitante and falta_target:
        return "Ningún usuario se encontró en ADManager."
    if falta_solicitante:
        return "El usuario solicitante no se encontró en ADManager."
    if falta_target:
        return "El usuario objetivo no se encontró en ADManager."
    return "No se encontró la información solicitada en ADManager."


def _resultado_429(op: OperacionReset) -> str:
    """Mensaje del estatus 429: se agotaron los tokens de ADManager.

    Input:
        op (OperacionReset): operación resuelta (no se consulta ningún campo).

    Output:
        str: mensaje legible para la columna `resultado_final`.
    """
    return "No se pudo ejecutar el reseteo: se agotaron los tokens de ADManager."


def _resultado_500(op: OperacionReset) -> str:
    """Mensaje del estatus 500: error inesperado, caso crítico que requiere revisión.

    Input:
        op (OperacionReset): operación resuelta (no se consulta ningún campo).

    Output:
        str: mensaje legible para la columna `resultado_final`.
    """
    return "Ocurrió un error inesperado en el proceso. Caso crítico: requiere revisión."


def _resultado_503(op: OperacionReset) -> str:
    """Mensaje del estatus 503, concatenando el error exacto que devolvió ADManager.

    La especificación pide incluir ese texto; se toma del bloque `| ADM-Raw response |`.

    Input:
        op (OperacionReset): operación resuelta; usa `adm_raw`.

    Output:
        str: mensaje base, más " Respuesta de ADManager: <detalle>" si hay `statusMessage`.
    """
    base = "Ocurrió un error en ADManager y el reseteo no se pudo ejecutar."
    detalle = mensaje_admanager(op.adm_raw)
    return f"{base} Respuesta de ADManager: {detalle}" if detalle else base


def _resultado_504(op: OperacionReset) -> str:
    """Mensaje del estatus 504: timeout (más de 35 s) en la comunicación con ADManager.

    Input:
        op (OperacionReset): operación resuelta (no se consulta ningún campo).

    Output:
        str: mensaje legible para la columna `resultado_final`.
    """
    return (
        "Timeout en la comunicación con ADManager "
        "(más de 35 segundos sin respuesta); se asume que el reseteo no se ejecutó."
    )


RESOLVERS = {
    200: _resultado_200,
    202: _resultado_202,
    403: _resultado_403,
    404: _resultado_404,
    429: _resultado_429,
    500: _resultado_500,
    503: _resultado_503,
    504: _resultado_504,
}


def resolver_resultado(op: OperacionReset) -> str:
    """Despacha la operación al resolver de su código HTTP y devuelve el mensaje humano.

    Añadir un código nuevo es añadir una función y una entrada en `RESOLVERS`: el resto del
    proceso no cambia.

    Input:
        op (OperacionReset): operación resuelta; el despacho se hace por `op.status`.

    Output:
        str: mensaje de la columna `resultado_final`; si el código no tiene resolver
        registrado, un texto explícito de "sin regla de negocio definida".
    """
    resolver = RESOLVERS.get(op.status)
    if resolver is None:
        return f"Estatus HTTP {op.status} sin regla de negocio definida."
    return resolver(op)

## 8. Capa `data/processed/` (staging)

Una fila por operación `resetuser`. **Sin `updated_at`**: el archivo depende sólo del
`.log` de entrada, así que regenerarlo produce siempre el mismo contenido.

In [31]:
def operacion_a_fila(op: OperacionReset) -> dict:
    """Convierte una `OperacionReset` en la fila de staging correspondiente.

    El `id` se calcula como `uuid5(UUID_NAMESPACE, operation_id)`: un UUID válido, único por
    registro y determinista (a diferencia de `uuid4`), que es lo que hace posible el upsert
    idempotente. La fila no lleva `updated_at`, porque el staging no depende del reloj.

    Input:
        op (OperacionReset): operación resuelta.

    Output:
        dict: una fila con las claves de `COLUMNAS_STAGING` (incluido `status_http`).
    """
    return {
        "id": str(uuid.uuid5(UUID_NAMESPACE, op.operation_id)),
        "timestamp": op.timestamp,
        "operation_id": op.operation_id,
        "solicitante": op.solicitante.sam,
        "target": op.target.sam,
        "accion": ACCION,
        "sistema": SISTEMA,
        "nombre_solicitante": op.solicitante.nombre_completo,
        "nombre_target": op.target.nombre_completo,
        "oficina_solicitante": op.solicitante.oficina,
        "oficina_target": op.target.oficina,
        "resultado_final": resolver_resultado(op),
        "status_http": op.status,
    }


def parsear_fecha(fecha: str) -> pd.DataFrame:
    """Procesa el `.log` de un día completo y devuelve su tabla de staging.

    Determinista: mismo log de entrada, mismo DataFrame de salida, byte por byte.

    Input:
        fecha (str): fecha en formato `YYYY-MM-DD`.

    Output:
        pd.DataFrame: una fila por operación `resetuser`, con las columnas
        `COLUMNAS_STAGING` y ordenada por (`timestamp`, `operation_id`).
    """
    filas = []
    for operacion in agrupar_operaciones(ruta_log(fecha)).values():
        op = enriquecer(operacion)
        if op is not None:
            filas.append(operacion_a_fila(op))
    df = pd.DataFrame(filas, columns=COLUMNAS_STAGING)
    return df.sort_values(["timestamp", "operation_id"]).reset_index(drop=True)


def escribir_csv_atomico(df: pd.DataFrame, destino: Path) -> Path:
    """Escribe un DataFrame a CSV de forma atómica: primero `.tmp` y luego rename.

    Así el archivo final nunca queda a medias si la ejecución se interrumpe.

    Input:
        df (pd.DataFrame): tabla a escribir.
        destino (Path): ruta final del CSV (su directorio se crea si falta).

    Output:
        Path: la misma ruta `destino`, ya con el archivo completo.
    """
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_name(destino.name + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    tmp.replace(destino)
    return destino


def ruta_staging(fecha: str) -> Path:
    """Devuelve la ruta del CSV de staging que corresponde a una fecha.

    Input:
        fecha (str): fecha en formato `YYYY-MM-DD`.

    Output:
        Path: `data/processed/resetuser_<fecha>.csv` (puede no existir todavía).
    """
    return PROCESSED_DIR / f"resetuser_{fecha}.csv"


def escribir_staging(fecha: str) -> tuple[pd.DataFrame, Path]:
    """Parsea el log de un día y persiste su staging en `data/processed/`.

    Input:
        fecha (str): fecha en formato `YYYY-MM-DD`.

    Output:
        tuple[pd.DataFrame, Path]: la tabla de staging y la ruta donde quedó escrita.
    """
    df = parsear_fecha(fecha)
    return df, escribir_csv_atomico(df, ruta_staging(fecha))

In [32]:
# Staging de todas las fechas disponibles
for _fecha in fechas_disponibles():
    _df, _destino = escribir_staging(_fecha)
    print(f"{_fecha}: {len(_df):>5} operaciones resetuser -> {_destino.relative_to(PROJECT_ROOT)}")

parsear_fecha(fechas_disponibles()[-1]).head()

2026-08-29:   433 operaciones resetuser -> data\processed\resetuser_2026-08-29.csv
2026-08-30:   144 operaciones resetuser -> data\processed\resetuser_2026-08-30.csv
2026-08-31:   621 operaciones resetuser -> data\processed\resetuser_2026-08-31.csv
2026-09-01:    40 operaciones resetuser -> data\processed\resetuser_2026-09-01.csv


,id,timestamp,operation_id,solicitante,target,accion,sistema,nombre_solicitante,nombre_target,oficina_solicitante,oficina_target,resultado_final,status_http
0,9066c9c9-aa28-5732-b0eb-21b5a114156a,2026-09-01T00:02:39.862105Z,b5ab6865ca8ac9629ff2eb303637d408,admsistemas520,520000228,reset_password,ADManager,Nombre_d216ff1cda Apellido_d216ff1cda,Nombre_86f0740bf6 Apellido_86f0740bf6,0520,0520,Proceso exitoso.,200
1,186d4641-7b14-5e2c-847f-e026156d6172,2026-09-01T00:03:40.445253Z,f7dde1e1995c0c89366858c6f494dc4d,admsistemas969,969000093,reset_password,ADManager,Nombre_b52d7f1bdc Apellido_b52d7f1bdc,Nombre_d1e1fa4a78 Apellido_d1e1fa4a78,0969,0969,Proceso exitoso.,200
2,004951bd-2aa0-5dbe-96b8-1a854b76587d,2026-09-01T00:04:10.348165Z,948f905fc0862cfd4668bd3788410f0a,gerencia626,626000368,reset_password,ADManager,Nombre_93fb1a3d4f Apellido_93fb1a3d4f,Nombre_06e1c133e5 Apellido_06e1c133e5,0626,0626,Proceso exitoso.,200
3,4f05207d-2dfd-5b97-bc70-b26d44e74aaa,2026-09-01T00:05:41.786904Z,c07941fefaffa21f57be8d15af0842f7,admsistemas876,1500244532,reset_password,ADManager,Nombre_fc0c595dbb Apellido_fc0c595dbb,Nombre_d762306d44 Apellido_d762306d44,0876,0876,Proceso exitoso.,200
4,17f940eb-1c17-59ac-b39d-1d2d310da708,2026-09-01T00:06:40.738316Z,e2870e64b55502fe2997677a0b4aa466,admsistemas969,969000010,reset_password,ADManager,Nombre_b52d7f1bdc Apellido_b52d7f1bdc,Nombre_06d84eae35 Apellido_06d84eae35,0969,0969,Proceso exitoso.,200


## 9. Upsert idempotente

El reporte sólo crece. Las filas cuyo `id` ya existe se descartan por completo —incluido
su `updated_at`, que conserva el valor de la carga original—. Sólo las filas nuevas reciben
la marca de tiempo de esta ejecución.

In [33]:
def cargar_reporte() -> pd.DataFrame:
    """Carga el reporte acumulado desde disco, o uno vacío si aún no existe.

    Todo se lee como `str` para que ids y timestamps no muten de tipo entre ejecuciones.

    Input:
        Ninguno. Lee `REPORT_PATH`.

    Output:
        pd.DataFrame: el reporte con las columnas `COLUMNAS_REPORTE` y sin valores `NaN`
        (los faltantes quedan como cadena vacía).
    """
    if REPORT_PATH.is_file():
        return pd.read_csv(REPORT_PATH, dtype=str).fillna("")
    return pd.DataFrame(columns=COLUMNAS_REPORTE, dtype=str)


def upsert_reporte(staging: pd.DataFrame, ahora: datetime | None = None) -> tuple[pd.DataFrame, int]:
    """Añade al reporte sólo las filas cuyo `id` aún no existe, y lo reescribe.

    Es el corazón de la idempotencia: las filas ya cargadas se descartan por completo, así
    que conservan su `updated_at` original y una reejecución no puede duplicar ni alterar
    datos. Sólo las filas nuevas reciben la marca de tiempo de esta ejecución.

    Input:
        staging (pd.DataFrame): tabla de staging de uno o varios días.
        ahora (datetime | None): marca de tiempo a usar como `updated_at`; `None` toma el
            UTC actual. Parametrizado para poder hacer pruebas reproducibles.

    Output:
        tuple[pd.DataFrame, int]: el reporte resultante y cuántas filas nuevas se añadieron.
        Si no hay filas nuevas devuelve `(reporte, 0)` y no toca el archivo.
    """
    reporte = cargar_reporte()
    conocidos = set(reporte["id"])
    nuevos = staging[~staging["id"].isin(conocidos)].copy()
    if nuevos.empty:
        return reporte, 0
    zona_horaria = ZoneInfo("America/Mexico_City")
    marca = (ahora or datetime.now(zona_horaria)).isoformat(timespec="seconds")
    nuevos["updated_at"] = marca
    final = pd.concat([reporte, nuevos[COLUMNAS_REPORTE]], ignore_index=True)
    final = final[COLUMNAS_REPORTE].sort_values(["timestamp", "id"]).reset_index(drop=True)
    escribir_csv_atomico(final, REPORT_PATH)
    return final, len(nuevos)


def procesar_fecha(fecha: str) -> tuple[pd.DataFrame, int]:
    """Ejecuta el proceso completo de un día: raw -> processed -> reporte.

    Input:
        fecha (str): fecha en formato `YYYY-MM-DD`.

    Output:
        tuple[pd.DataFrame, int]: el reporte acumulado y el número de registros nuevos.
    """
    staging, _ = escribir_staging(fecha)
    return upsert_reporte(staging)

In [35]:
# Ejecución end-to-end de todos los días disponibles
for _fecha in fechas_disponibles():
    _reporte, _nuevos = procesar_fecha(_fecha)
    print(f"{_fecha}: +{_nuevos:>4} registros nuevos | total acumulado: {len(_reporte)}")

_reporte = cargar_reporte()
print()
print("Reporte:", REPORT_PATH.relative_to(PROJECT_ROOT), "|", len(_reporte), "filas")
_reporte.head()

2026-08-29: + 433 registros nuevos | total acumulado: 433
2026-08-30: + 144 registros nuevos | total acumulado: 577
2026-08-31: + 621 registros nuevos | total acumulado: 1198
2026-09-01: +  40 registros nuevos | total acumulado: 1238

Reporte: reports\tabla_reporte_bot.csv | 1238 filas


,id,updated_at,timestamp,operation_id,solicitante,target,accion,sistema,nombre_solicitante,nombre_target,oficina_solicitante,oficina_target,resultado_final
0,0f629a41-89f6-515d-bdc2-a238898d8203,2026-09-06T18:43:22-06:00,2026-08-29T12:59:24.457383Z,b192579b98e2f5c469a96eb7e19241cf,admsistemas970,1500335663,reset_password,ADManager,Nombre_e309a3c6d9 Apellido_e309a3c6d9,Nombre_d2c3f50cbe Apellido_d2c3f50cbe,0970,0970,Proceso exitoso.
1,d32cdbaf-7f43-5163-9e8a-d1a777aa8717,2026-09-06T18:43:22-06:00,2026-08-29T13:01:12.563198Z,af509cb31490c15475855c84a5181dbb,admsistemas315,304000282,reset_password,ADManager,Nombre_025cc5a075 Apellido_025cc5a075,Nombre_56132d9123 Apellido_56132d9123,0315,0315,Proceso exitoso.
2,261f7a8e-d12d-5ddb-8d6d-f764343e0a57,2026-09-06T18:43:22-06:00,2026-08-29T13:09:33.514331Z,1fc53122d945523bc92aa35e8b723bfc,admsistemas604,604000404,reset_password,ADManager,Nombre_9c4f943381 Apellido_9c4f943381,Nombre_e076e45146 Apellido_e076e45146,0604,0604,Proceso exitoso.
3,ffe977e3-2892-594d-9e09-1c893faa081d,2026-09-06T18:43:22-06:00,2026-08-29T13:11:05.963642Z,77c2e7e1f63bc16e2124673c539a41a5,admsistemas212,212000892,reset_password,ADManager,Nombre_eb542d8ff7 Apellido_eb542d8ff7,Nombre_97e61be581 Apellido_97e61be581,0212,0212,Proceso exitoso.
4,0832e1e4-186a-5848-90fa-63d52173a6ea,2026-09-06T18:43:22-06:00,2026-08-29T13:11:29.033395Z,18dc5c09eaf48ee830bf14d33cac3eab,admsistemas212,212001188,reset_password,ADManager,Nombre_eb542d8ff7 Apellido_eb542d8ff7,Nombre_d5d2b98bb4 Apellido_d5d2b98bb4,0212,0212,Proceso exitoso.


## 10. Validaciones

La prueba que importa es la de **idempotencia**: reprocesar todo por segunda vez no debe
cambiar ni un byte del reporte. Esa es la definición ejecutable del requisito.

In [36]:
def hash_archivo(path: Path) -> str:
    """Calcula el SHA-256 de un archivo, para comprobar que no cambió ni un byte.

    Es la verificación ejecutable del requisito de idempotencia.

    Input:
        path (Path): ruta del archivo, que se lee completo en memoria.

    Output:
        str: digest SHA-256 en hexadecimal.
    """
    return hashlib.sha256(path.read_bytes()).hexdigest()


_antes = hash_archivo(REPORT_PATH)
_filas_antes = len(cargar_reporte())

# Reejecución completa, incluyendo días anteriores (escenario de recuperación)
for _fecha in fechas_disponibles():
    procesar_fecha(_fecha)

_despues = hash_archivo(REPORT_PATH)
_filas_despues = len(cargar_reporte())

print(f"Filas antes / después : {_filas_antes} / {_filas_despues}")
print(f"SHA-256 idéntico      : {_antes == _despues}")
assert _antes == _despues, "El proceso NO es idempotente"
assert _filas_antes == _filas_despues
print("\nIdempotencia verificada.")

Filas antes / después : 1238 / 1238
SHA-256 idéntico      : True

Idempotencia verificada.


In [37]:
_rep = cargar_reporte()

print("Unicidad de id            :", _rep["id"].is_unique)
print("Unicidad de operation_id  :", _rep["operation_id"].is_unique)
print("Filas sin resultado_final :", int((_rep["resultado_final"] == "").sum()))
print("Sistemas presentes        :", _rep["sistema"].unique().tolist())
print("Acciones presentes        :", _rep["accion"].unique().tolist())
print()

_staging = pd.concat([parsear_fecha(f) for f in fechas_disponibles()], ignore_index=True)
print("Distribución de status HTTP:")
print(_staging["status_http"].value_counts().sort_index().to_string())
print()
print("Distribución de resultado_final:")
print(_rep["resultado_final"].value_counts().to_string())

Unicidad de id            : True
Unicidad de operation_id  : True
Filas sin resultado_final : 0
Sistemas presentes        : ['ADManager']
Acciones presentes        : ['reset_password']

Distribución de status HTTP:
status_http
200    1108
403      41
404      79
500       1
503       3
504       6

Distribución de resultado_final:
resultado_final
Proceso exitoso.                                                                                                                                                                                                                        1108
El usuario objetivo no se encontró en ADManager.                                                                                                                                                                                          79
El usuario solicitante y el usuario objetivo no pertenecen a la misma oficina.                                                                                                   

In [38]:
# Cobertura: qué reglas de negocio no tienen ningún caso en los datos actuales
_presentes = set(_staging["status_http"].unique())
_sin_datos = sorted(set(RESOLVERS) - _presentes)
print("Códigos implementados sin casos en los logs:", _sin_datos)

# Ningún 403 debería caer en el mensaje genérico: si cae, hay una causa que no modelamos.
_genericos = _staging[
    (_staging["status_http"] == 403)
    & (_staging["resultado_final"] == "El reseteo fue rechazado por las validaciones de acceso del bot.")
]
print(f"403 sin causa identificada : {len(_genericos)} de {(_staging['status_http'] == 403).sum()}")
if len(_genericos):
    print(_genericos[["operation_id", "oficina_solicitante", "oficina_target"]].to_string(index=False))

# Filas con información incompleta de ADManager (esperable en 404, 500 y 504)
_incompletas = _rep[(_rep["nombre_target"] == "") | (_rep["oficina_target"] == "")]
print(f"\nFilas sin datos del target: {len(_incompletas)}")
_incompletas.merge(_staging[["id", "status_http"]], on="id")["status_http"].value_counts().sort_index()

Códigos implementados sin casos en los logs: [202, 429]
403 sin causa identificada : 0 de 41

Filas sin datos del target: 82


status_http
403     3
404    79
Name: count, dtype: int64

## 11. Zona de experimentación

Utilidades para inspeccionar casos concretos: ver el log crudo de una operación, revisar
cómo quedó clasificada y probar los resolvers contra escenarios sintéticos (útil para el
`202` y el `429`, que no tienen datos reales).

In [39]:
def buscar_operacion(operation_id: str) -> Operation | None:
    """Recupera una operación cruda por su `operation_Id`, buscando en todos los días.

    Input:
        operation_id (str): identificador de la operación tal como aparece en el log.

    Output:
        Operation | None: la operación con sus registros, o `None` si no está en ningún log.
    """
    for fecha in fechas_disponibles():
        operaciones = agrupar_operaciones(ruta_log(fecha))
        if operation_id in operaciones:
            return operaciones[operation_id]
    return None


def ver_crudo(operation_id: str, limite_por_linea: int = 300) -> None:
    """Imprime el log original de una operación, recortando las líneas muy largas.

    Utilidad de inspección manual: sirve para auditar por qué una fila quedó clasificada
    como quedó.

    Input:
        operation_id (str): identificador de la operación.
        limite_por_linea (int): máximo de caracteres a imprimir por línea; lo que sobra se
            marca con "[…]".

    Output:
        None. Escribe en la salida estándar.
    """
    operacion = buscar_operacion(operation_id)
    if operacion is None:
        print("No encontrada:", operation_id)
        return
    for linea in operacion.texto_crudo.splitlines():
        print(linea[:limite_por_linea] + (" […]" if len(linea) > limite_por_linea else ""))


def muestra_por_status(status: int, n: int = 3) -> pd.DataFrame:
    """Devuelve una muestra de filas del reporte para un código HTTP dado.

    Input:
        status (int): código HTTP a filtrar (se busca en `_staging`).
        n (int): número máximo de filas a devolver.

    Output:
        pd.DataFrame: hasta `n` filas indexadas por `id`, con las columnas de interés
        (timestamp, usuarios, oficinas y `resultado_final`).
    """
    ids = _staging.loc[_staging["status_http"] == status, "id"]
    columnas = ["timestamp", "solicitante", "target", "oficina_solicitante",
                "oficina_target", "resultado_final"]
    return cargar_reporte().set_index("id").loc[ids, columnas].head(n)


muestra_por_status(404)

,timestamp,solicitante,target,oficina_solicitante,oficina_target,resultado_final
id,,,,,,
248eaf94-f0ce-5c8c-a2e4-d20c1516a9a9,2026-08-29T13:53:39.848761Z,admsistemas530,SUPMERMAS,0530,,El usuario objetivo no se encontró en ADManager.
f6741c60-4e20-5479-a419-8d9491ff7b8d,2026-08-29T13:57:22.007434Z,gerencia506,TRANSNETWORK,0506,,El usuario objetivo no se encontró en ADManager.
7910b47a-18d7-5c9b-8967-3534d529fac8,2026-08-29T15:09:33.201248Z,admsistemas946,1800282232,0946,,El usuario objetivo no se encontró en ADManager.


In [40]:
# Un caso de cada código presente en los datos
for _status in sorted(_staging["status_http"].unique()):
    _fila = _staging[_staging["status_http"] == _status].iloc[0]
    print(f"[{_status}] {_fila['solicitante']} -> {_fila['target']}")
    print(f"      {_fila['resultado_final']}")
    print(f"      operation_id: {_fila['operation_id']}")
    print()

[200] admsistemas970 -> 1500335663
      Proceso exitoso.
      operation_id: b192579b98e2f5c469a96eb7e19241cf

[403] admsistemas062 -> 1500023149
      El usuario solicitante y el usuario objetivo no pertenecen a la misma oficina.
      operation_id: 895db57ed39c8499d426c89d9f924394

[404] admsistemas530 -> SUPMERMAS
      El usuario objetivo no se encontró en ADManager.
      operation_id: 21c515e3be1d2adcff57ce6957104d0c

[500] admsistemas374 -> 374000417
      Ocurrió un error inesperado en el proceso. Caso crítico: requiere revisión.
      operation_id: 610113353adb8a0c1c67a3d3b5c41989

[503] admsistemas507 -> sgsupmercado507
      Ocurrió un error en ADManager y el reseteo no se pudo ejecutar. Respuesta de ADManager: LOGON_NAME: sgsupmercado507@retailstore.com - No such user matched. Verify the LDAP attribute in search query or could be a privilege issue.
      operation_id: c6f5a3a95ed29e480cd335c3b0930812

[504] AdmSistemas014 -> 14241095
      Timeout en la comunicación con AD

In [41]:
# Probar los resolvers contra escenarios sintéticos (202 y 429 no existen en los logs)
def escenario(status: int, **kwargs) -> str:
    """Construye una operación sintética y devuelve el mensaje que produciría.

    Sirve para probar los resolvers sin datos reales, sobre todo los del 202 y el 429, que
    no aparecen en los logs actuales.

    Input:
        status (int): código HTTP a simular.
        **kwargs: campos a sobreescribir, todos con valor por defecto:
            `sam_sol`/`sam_tgt` (str), `sol_encontrado`/`tgt_encontrado` (bool),
            `oficina_sol`/`oficina_tgt` (str), `desc_sol` (str), `ou_tgt` (str),
            `adm_raw` (dict | None).

    Output:
        str: el mensaje de `resultado_final` que devolvería el resolver de ese estatus.
    """
    solicitante = Usuario(
        sam=kwargs.get("sam_sol", "gerencia001"),
        encontrado=kwargs.get("sol_encontrado", True),
        oficina=kwargs.get("oficina_sol", "0001"),
        descripcion=kwargs.get("desc_sol", "Gerente Tienda"),
    )
    target = Usuario(
        sam=kwargs.get("sam_tgt", "001000001"),
        encontrado=kwargs.get("tgt_encontrado", True),
        oficina=kwargs.get("oficina_tgt", "0001"),
        ou_name=kwargs.get("ou_tgt", "Tienda/POS/Intelexion Tienda"),
    )
    op = OperacionReset("sintetico", "2026-09-01T00:00:00Z", status, solicitante, target,
                        kwargs.get("adm_raw"))
    return resolver_resultado(op)


print("202 corporativo :", escenario(202, oficina_tgt="CORPORATIVO"))
print("403 oficinas    :", escenario(403, oficina_tgt="0999"))
print("403 privilegios :", escenario(403, desc_sol="Surtidor Ropa Y Variedades"))
print("403 cedis BY    :", escenario(403, ou_tgt="OAT/Cedis/BY"))
print("404 ambos       :", escenario(404, sol_encontrado=False, tgt_encontrado=False))
print("404 target      :", escenario(404, tgt_encontrado=False))
print("429 tokens      :", escenario(429))
print("500 interno     :", escenario(500))
print("503 admanager   :", escenario(503, adm_raw={"status": 200,
      "body": [{"statusMessage": "LOGON_NAME: x - No such user matched.", "status": "0"}]}))
print("504 timeout     :", escenario(504))

202 corporativo : El usuario objetivo pertenece a Corporativo, por lo que puede autoresetear su contraseña sin el bot.
403 oficinas    : El usuario solicitante y el usuario objetivo no pertenecen a la misma oficina.
403 privilegios : El usuario solicitante no es gerente ni administrador de sistemas.
403 cedis BY    : El usuario objetivo pertenece a la OU 'OAT/Cedis/BY' y no puede ser reseteado mediante el bot.
404 ambos       : Ningún usuario se encontró en ADManager.
404 target      : El usuario objetivo no se encontró en ADManager.
429 tokens      : No se pudo ejecutar el reseteo: se agotaron los tokens de ADManager.
500 interno     : Ocurrió un error inesperado en el proceso. Caso crítico: requiere revisión.
503 admanager   : Ocurrió un error en ADManager y el reseteo no se pudo ejecutar. Respuesta de ADManager: LOGON_NAME: x - No such user matched.
504 timeout     : Timeout en la comunicación con ADManager (más de 35 segundos sin respuesta); se asume que el reseteo no se ejecutó.


In [ ]:
# Reconstrucción desde cero.
#
# El upsert es append-only por diseño: si cambias una regla de negocio, las filas que ya
# están en el reporte NO se recalculan. Para ver el efecto de un cambio hay que borrar las
# salidas y regenerarlas. Esto es exactamente lo que se quiere en producción (el histórico
# no se reescribe solo) y por eso la reconstrucción es un acto explícito.
def reconstruir_todo(confirmar: bool = False) -> pd.DataFrame | None:
    """Borra las salidas y regenera el reporte desde cero a partir de los `.log`.

    El upsert es append-only por diseño: cambiar una regla de negocio no recalcula las filas
    ya cargadas. Por eso la reconstrucción es un acto explícito y protegido por `confirmar`.

    Input:
        confirmar (bool): `False` (por defecto) sólo informa qué se borraría; `True` borra
            el reporte y los CSV de staging y vuelve a procesar todas las fechas.

    Output:
        pd.DataFrame | None: el reporte reconstruido, o `None` si no se confirmó.
    """
    if not confirmar:
        print("Llama a reconstruir_todo(confirmar=True) para borrar y regenerar.")
        print(f"  Se borrarían: {REPORT_PATH.name} y {len(list(PROCESSED_DIR.glob('resetuser_*.csv')))} archivos de staging")
        return None
    REPORT_PATH.unlink(missing_ok=True)
    for archivo in PROCESSED_DIR.glob("resetuser_*.csv"):
        archivo.unlink()
    for fecha in fechas_disponibles():
        _, nuevos = procesar_fecha(fecha)
        print(f"{fecha}: +{nuevos} registros")
    return cargar_reporte()


reconstruir_todo()

## 12. Modulazción

La migración a `src/` es: cada sección pasa a su módulo según la tabla de la sección 1,
las celdas de comprobación se convierten en tests con fixtures pequeños, y `main.py`
queda reducido a la orquestación:

```bash
python main.py                                     # el .log más reciente
python main.py --date 2026-08-30                   # reprocesa un día pasado
python main.py --from 2026-08-29 --to 2026-08-31   # recuperación de varios días
python main.py --date 2026-08-30 --dry-run         # reporta sin escribir
```